In [1]:
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime

In [2]:
conn = pyodbc.connect(
    'Driver={ODBC Driver 17 for SQL Server};'
    'Server=PUGSQLRSVRAG02.ug.sbicdirectory.com,5200;'
    'UID=RSVR_P_SQLACC_R_DS;'
    'Database=RSVRRealtimeFinacleUGPRD;'
    'PWD=M5tekitu45t65!g;'
    'Trusted_Connection=no;'
)

In [4]:
query = """SELECT
            hd.TRAN_DATE,
            hd.RCRE_TIME,
            hd.TRAN_ID,
            hd.TRAN_AMT,
            hd.TRAN_CRNCY_CODE,
            hd.PART_TRAN_TYPE,
            hd.TRAN_SUB_TYPE,
            hd.TRAN_TYPE,
            hd.TRAN_PARTICULAR,
            hd.TRAN_RMKS,
            hd.RPT_CODE,
            hd.ENTRY_USER_ID,
            gm.FORACID,
            gm.ACID,
            gm.ACCT_OPN_DATE
        FROM RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd
        JOIN RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm
            ON hd.ACID = gm.ACID
        WHERE hd.ACID='UG6246002'
        ORDER BY hd.RCRE_TIME;
        """

In [5]:
df = pd.read_sql(query, conn)

/tmp/ipykernel_3282199/1168034203.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [7]:
df

,TRAN_DATE,RCRE_TIME,TRAN_ID,TRAN_AMT,TRAN_CRNCY_CODE,PART_TRAN_TYPE,TRAN_SUB_TYPE,TRAN_TYPE,TRAN_PARTICULAR,TRAN_RMKS,RPT_CODE,ENTRY_USER_ID,FORACID,ACID,ACCT_OPN_DATE
0,2025-11-04,2025-11-04 16:47:10,0378139,100.00,USD,C,RI,C,ASHIRAF,TEXTILE BUSINESS,CMI,266331,9030027282317,UG6246002,2025-11-04
1,2025-11-04,2025-11-04 18:56:32,S99211978,10.00,USD,D,BI,T,FEE - CASH DEPOSIT,FEE - CASH DEPOSIT,TRF,164751,9030027282317,UG6246002,2025-11-04
2,2025-11-04,2025-11-04 18:56:32,S99211978,1.50,USD,D,BI,T,EXCISE DUTY FEE - CASH DEPOSIT,Access Fee Collection,TRF,164751,9030027282317,UG6246002,2025-11-04
3,2025-11-04,2025-11-05 02:04:49,S99264131,88.50,USD,D,BI,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04
4,2025-11-04,2025-11-05 02:04:49,S99264131,88.50,USD,C,BI,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,2026-07-05,2026-07-05 12:00:54,S63842363,2.81,USD,D,BI,T,GOVERNMENT EXCISE DUTY CHARGE,EXCISE DUTY,CHG,SYSTEM,9030027282317,UG6246002,2025-11-04
64,2026-07-06,2026-07-06 19:02:52,03105256,670000.00,USD,D,PI,C,ASIIMWE MUTUNGI,LOAN REPAYMENT,CLR,254195,9030027282317,UG6246002,2025-11-04
65,2026-07-06,2026-07-06 19:41:20,03109660,1600.00,USD,C,RI,C,ASIIMWE MUTUNGI,LOAN REPAYMENT,CMI,254195,9030027282317,UG6246002,2025-11-04
66,2026-07-06,2026-07-06 20:18:16,S64129960,10.00,USD,D,BI,T,FEE - CASH DEPOSIT,FEE - CASH DEPOSIT,TRF,164751,9030027282317,UG6246002,2025-11-04


In [8]:
df['TRAN_DATE'] = pd.to_datetime(df['TRAN_DATE'])
df['RCRE_TIME'] = pd.to_datetime(df['RCRE_TIME'])

df = df.sort_values('RCRE_TIME').reset_index(drop=True)

In [9]:
fee_patterns = [
    'FEE',
    'STAMP DUTY',
    'EXCISE',
    'SMS',
    'COMMISSION',
    'CHARGE',
    'INTEREST'
]

mask = (
    ~df['TRAN_PARTICULAR'].fillna('').str.upper().str.contains('|'.join(fee_patterns))
    &
    (df['RPT_CODE'] != 'CHG')
)

txns = df.loc[mask].copy()

In [10]:
txns

,TRAN_DATE,RCRE_TIME,TRAN_ID,TRAN_AMT,TRAN_CRNCY_CODE,PART_TRAN_TYPE,TRAN_SUB_TYPE,TRAN_TYPE,TRAN_PARTICULAR,TRAN_RMKS,RPT_CODE,ENTRY_USER_ID,FORACID,ACID,ACCT_OPN_DATE
0,2025-11-04,2025-11-04 16:47:10,0378139,100.00,USD,C,RI,C,ASHIRAF,TEXTILE BUSINESS,CMI,266331,9030027282317,UG6246002,2025-11-04
3,2025-11-04,2025-11-05 02:04:49,S99264131,88.50,USD,D,BI,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04
4,2025-11-04,2025-11-05 02:04:49,S99264131,88.50,USD,C,BI,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04
16,2026-03-26,2026-03-26 10:30:09,S35728440,103.00,USD,C,CI,T,JI,BOL 87318025 KEYSTONE TEXTILES,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04
24,2026-03-30,2026-03-30 19:23:28,S37125360,99985.00,USD,C,CI,T,TERENCE A WALTS,IT26089UG0175304,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04
27,2026-03-31,2026-03-31 09:31:27,S37340109,99900.15,USD,D,CI,T,payment for,OT26090UG0109567,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04
29,2026-03-31,2026-04-01 02:31:52,S37632727,1.61,USD,D,IC,T,9030027282317:Int.Coll:01-03-2026 to 31-03-2026,Interest run,INT,UGCTRLM,9030027282317,UG6246002,2025-11-04
30,2026-04-01,2026-04-01 08:01:20,S37707558,350000.00,USD,C,CI,T,1/WEI FU,IT26091UG0175981,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04
31,2026-04-01,2026-04-01 09:52:19,S37744350,140237.94,USD,D,CI,T,dsaf,PAYEX-87530572383225,TRF,UGMQMPROC,9030027282317,UG6246002,2025-11-04
33,2026-04-01,2026-04-01 10:01:20,S37746899,150001.41,USD,D,CI,T,ADFGER,OT26091UG0110373,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04


### Prepare credits and debits

In [11]:
credits = txns[txns['PART_TRAN_TYPE']=='C'].copy()
debits = txns[txns['PART_TRAN_TYPE']=='D'].copy()

credits = credits.sort_values('RCRE_TIME')
debits = debits.sort_values('RCRE_TIME')

In [21]:
credits[['RCRE_TIME','TRAN_AMT','TRAN_PARTICULAR']]

,RCRE_TIME,TRAN_AMT,TRAN_PARTICULAR
0,2025-11-04 16:47:10,100.0,ASHIRAF
4,2025-11-05 02:04:49,88.5,Ac xfr from gl 0603 to 3500
16,2026-03-26 10:30:09,103.0,JI
24,2026-03-30 19:23:28,99985.0,TERENCE A WALTS
30,2026-04-01 08:01:20,350000.0,1/WEI FU
36,2026-04-01 11:02:43,17000.0,MIE SUGIYAMA
39,2026-04-02 11:43:12,46970.0,1/JOE PURCELL
47,2026-04-07 08:04:29,230307.5,1/PERAZZINI PIERGIUSEPPE
48,2026-04-07 08:04:59,84997.0,CHARLES RAHILL LLC
50,2026-04-15 17:00:42,500000.0,JAMES R SMITH


In [22]:
debits[['RCRE_TIME','TRAN_AMT','TRAN_PARTICULAR']]

,RCRE_TIME,TRAN_AMT,TRAN_PARTICULAR
3,2025-11-05 02:04:49,88.50,Ac xfr from gl 0603 to 3500
27,2026-03-31 09:31:27,99900.15,payment for
29,2026-04-01 02:31:52,1.61,9030027282317:Int.Coll:01-03-2026 to 31-03-2026
31,2026-04-01 09:52:19,140237.94,dsaf
33,2026-04-01 10:01:20,150001.41,ADFGER
35,2026-04-01 10:14:35,9500.16,ADFSG
42,2026-04-04 09:53:34,113000.00,NALUNKUMA JEMIMAH RUTH
61,2026-06-26 15:55:31,145307.52,URA AGENCY NOTICE 2260019422449
64,2026-07-06 19:02:52,670000.00,ASIIMWE MUTUNGI


In [17]:
debits['DEBIT_TIME'] = debits['RCRE_TIME']

In [18]:
velocity = pd.merge_asof(
    credits.sort_values('RCRE_TIME'),
    debits,
    on='RCRE_TIME',
    direction='forward',
    suffixes=('_credit', '_debit')
)

velocity['hours_to_debit'] = (
    velocity['DEBIT_TIME'] - velocity['RCRE_TIME']
).dt.total_seconds() / 3600

In [19]:
velocity

,TRAN_DATE_credit,RCRE_TIME,TRAN_ID_credit,TRAN_AMT_credit,TRAN_CRNCY_CODE_credit,PART_TRAN_TYPE_credit,TRAN_SUB_TYPE_credit,TRAN_TYPE_credit,TRAN_PARTICULAR_credit,TRAN_RMKS_credit,...,TRAN_TYPE_debit,TRAN_PARTICULAR_debit,TRAN_RMKS_debit,RPT_CODE_debit,ENTRY_USER_ID_debit,FORACID_debit,ACID_debit,ACCT_OPN_DATE_debit,DEBIT_TIME,hours_to_debit
0,2025-11-04,2025-11-04 16:47:10,0378139,100.0,USD,C,RI,C,ASHIRAF,TEXTILE BUSINESS,...,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04,2025-11-05 02:04:49,9.294167
1,2025-11-04,2025-11-05 02:04:49,S99264131,88.5,USD,C,BI,T,Ac xfr from gl 0603 to 3500,TACBSH,...,T,Ac xfr from gl 0603 to 3500,TACBSH,TRF,UGCTRLM,9030027282317,UG6246002,2025-11-04,2025-11-05 02:04:49,0.000000
2,2026-03-26,2026-03-26 10:30:09,S35728440,103.0,USD,C,CI,T,JI,BOL 87318025 KEYSTONE TEXTILES,...,T,payment for,OT26090UG0109567,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04,2026-03-31 09:31:27,119.021667
3,2026-03-30,2026-03-30 19:23:28,S37125360,99985.0,USD,C,CI,T,TERENCE A WALTS,IT26089UG0175304,...,T,payment for,OT26090UG0109567,TRF,UGFIVUSR,9030027282317,UG6246002,2025-11-04,2026-03-31 09:31:27,14.133056
4,2026-04-01,2026-04-01 08:01:20,S37707558,350000.0,USD,C,CI,T,1/WEI FU,IT26091UG0175981,...,T,dsaf,PAYEX-87530572383225,TRF,UGMQMPROC,9030027282317,UG6246002,2025-11-04,2026-04-01 09:52:19,1.849722
5,2026-04-01,2026-04-01 11:02:43,S37768270,17000.0,USD,C,CI,T,MIE SUGIYAMA,IT26091UG0176129,...,C,NALUNKUMA JEMIMAH RUTH,0703983068,CLR,249214,9030027282317,UG6246002,2025-11-04,2026-04-04 09:53:34,70.847500
6,2026-04-02,2026-04-02 11:43:12,S38127338,46970.0,USD,C,CI,T,1/JOE PURCELL,IT26091UG0176222,...,C,NALUNKUMA JEMIMAH RUTH,0703983068,CLR,249214,9030027282317,UG6246002,2025-11-04,2026-04-04 09:53:34,46.172778
7,2026-04-07,2026-04-07 08:04:29,S39188230,230307.5,USD,C,CI,T,1/PERAZZINI PIERGIUSEPPE,IT26097UG0177536,...,T,URA AGENCY NOTICE 2260019422449,41189721,TRF,249578,9030027282317,UG6246002,2025-11-04,2026-06-26 15:55:31,1927.850556
8,2026-04-07,2026-04-07 08:04:59,S39188586,84997.0,USD,C,CI,T,CHARLES RAHILL LLC,IT26097UG0177614,...,T,URA AGENCY NOTICE 2260019422449,41189721,TRF,249578,9030027282317,UG6246002,2025-11-04,2026-06-26 15:55:31,1927.842222
9,2026-04-15,2026-04-15 17:00:42,03109689,500000.0,USD,C,BI,T,JAMES R SMITH,IT26085UG0172709,...,T,URA AGENCY NOTICE 2260019422449,41189721,TRF,249578,9030027282317,UG6246002,2025-11-04,2026-06-26 15:55:31,1726.913611


In [20]:
velocity[[ 'RCRE_TIME', 'TRAN_AMT_credit', 'DEBIT_TIME', 'TRAN_AMT_debit', 'hours_to_debit']]

,RCRE_TIME,TRAN_AMT_credit,DEBIT_TIME,TRAN_AMT_debit,hours_to_debit
0,2025-11-04 16:47:10,100.0,2025-11-05 02:04:49,88.50,9.294167
1,2025-11-05 02:04:49,88.5,2025-11-05 02:04:49,88.50,0.000000
2,2026-03-26 10:30:09,103.0,2026-03-31 09:31:27,99900.15,119.021667
3,2026-03-30 19:23:28,99985.0,2026-03-31 09:31:27,99900.15,14.133056
4,2026-04-01 08:01:20,350000.0,2026-04-01 09:52:19,140237.94,1.849722
5,2026-04-01 11:02:43,17000.0,2026-04-04 09:53:34,113000.00,70.847500
6,2026-04-02 11:43:12,46970.0,2026-04-04 09:53:34,113000.00,46.172778
7,2026-04-07 08:04:29,230307.5,2026-06-26 15:55:31,145307.52,1927.850556
8,2026-04-07 08:04:59,84997.0,2026-06-26 15:55:31,145307.52,1927.842222
9,2026-04-15 17:00:42,500000.0,2026-06-26 15:55:31,145307.52,1726.913611


In [23]:
velocity['hours_to_debit'].describe()

count      10.000000
mean      584.392528
std       883.294009
min         0.000000
25%        10.503889
50%        58.510139
75%      1324.940625
max      1927.850556
Name: hours_to_debit, dtype: float64

In [24]:
timeline = txns[
    ['RCRE_TIME',
     'PART_TRAN_TYPE',
     'TRAN_AMT',
     'TRAN_PARTICULAR',
     'TRAN_RMKS']
].sort_values('RCRE_TIME')

timeline.head(50)

,RCRE_TIME,PART_TRAN_TYPE,TRAN_AMT,TRAN_PARTICULAR,TRAN_RMKS
0,2025-11-04 16:47:10,C,100.00,ASHIRAF,TEXTILE BUSINESS
3,2025-11-05 02:04:49,D,88.50,Ac xfr from gl 0603 to 3500,TACBSH
4,2025-11-05 02:04:49,C,88.50,Ac xfr from gl 0603 to 3500,TACBSH
16,2026-03-26 10:30:09,C,103.00,JI,BOL 87318025 KEYSTONE TEXTILES
24,2026-03-30 19:23:28,C,99985.00,TERENCE A WALTS,IT26089UG0175304
27,2026-03-31 09:31:27,D,99900.15,payment for,OT26090UG0109567
29,2026-04-01 02:31:52,D,1.61,9030027282317:Int.Coll:01-03-2026 to 31-03-2026,Interest run
30,2026-04-01 08:01:20,C,350000.00,1/WEI FU,IT26091UG0175981
31,2026-04-01 09:52:19,D,140237.94,dsaf,PAYEX-87530572383225
33,2026-04-01 10:01:20,D,150001.41,ADFGER,OT26091UG0110373


In [25]:
timeline[
    (timeline['RCRE_TIME'] >= '2026-03-25')
    & (timeline['RCRE_TIME'] <= '2026-04-10')
]

,RCRE_TIME,PART_TRAN_TYPE,TRAN_AMT,TRAN_PARTICULAR,TRAN_RMKS
16,2026-03-26 10:30:09,C,103.00,JI,BOL 87318025 KEYSTONE TEXTILES
24,2026-03-30 19:23:28,C,99985.00,TERENCE A WALTS,IT26089UG0175304
27,2026-03-31 09:31:27,D,99900.15,payment for,OT26090UG0109567
29,2026-04-01 02:31:52,D,1.61,9030027282317:Int.Coll:01-03-2026 to 31-03-2026,Interest run
30,2026-04-01 08:01:20,C,350000.00,1/WEI FU,IT26091UG0175981
31,2026-04-01 09:52:19,D,140237.94,dsaf,PAYEX-87530572383225
33,2026-04-01 10:01:20,D,150001.41,ADFGER,OT26091UG0110373
35,2026-04-01 10:14:35,D,9500.16,ADFSG,PAYEX-87532607383791
36,2026-04-01 11:02:43,C,17000.00,MIE SUGIYAMA,IT26091UG0176129
39,2026-04-02 11:43:12,C,46970.00,1/JOE PURCELL,IT26091UG0176222


In [26]:
import re

def clean_counterparty(text):
    if pd.isna(text):
        return None

    text = str(text).strip().upper()

    # Remove leading numbering like "1/"
    text = re.sub(r'^\d+/', '', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text

txns['COUNTERPARTY'] = txns['TRAN_PARTICULAR'].apply(clean_counterparty)

In [28]:
txns[['TRAN_PARTICULAR','COUNTERPARTY']].head(15)

,TRAN_PARTICULAR,COUNTERPARTY
0,ASHIRAF,ASHIRAF
3,Ac xfr from gl 0603 to 3500,AC XFR FROM GL 0603 TO 3500
4,Ac xfr from gl 0603 to 3500,AC XFR FROM GL 0603 TO 3500
16,JI,JI
24,TERENCE A WALTS,TERENCE A WALTS
27,payment for,PAYMENT FOR
29,9030027282317:Int.Coll:01-03-2026 to 31-03-2026,9030027282317:INT.COLL:01-03-2026 TO 31-03-2026
30,1/WEI FU,WEI FU
31,dsaf,DSAF
33,ADFGER,ADFGER


In [29]:
incoming = txns[txns['PART_TRAN_TYPE'] == 'C'].copy()
outgoing = txns[txns['PART_TRAN_TYPE'] == 'D'].copy()

In [30]:
incoming_summary = (
    incoming
    .groupby('COUNTERPARTY')
    .agg(
        Transactions=('TRAN_AMT', 'count'),
        Total_Received=('TRAN_AMT', 'sum'),
        Average_Received=('TRAN_AMT', 'mean')
    )
    .sort_values('Total_Received', ascending=False)
)

incoming_summary

,Transactions,Total_Received,Average_Received
COUNTERPARTY,,,
JAMES R SMITH,1,500000.0,500000.0
WEI FU,1,350000.0,350000.0
PERAZZINI PIERGIUSEPPE,1,230307.5,230307.5
TERENCE A WALTS,1,99985.0,99985.0
CHARLES RAHILL LLC,1,84997.0,84997.0
JOE PURCELL,1,46970.0,46970.0
MIE SUGIYAMA,1,17000.0,17000.0
ASIIMWE MUTUNGI,1,1600.0,1600.0
JI,1,103.0,103.0


In [31]:
outgoing_summary = (
    outgoing
    .groupby('COUNTERPARTY')
    .agg(
        Transactions=('TRAN_AMT', 'count'),
        Total_Paid=('TRAN_AMT', 'sum'),
        Average_Paid=('TRAN_AMT', 'mean')
    )
    .sort_values('Total_Paid', ascending=False)
)

outgoing_summary

,Transactions,Total_Paid,Average_Paid
COUNTERPARTY,,,
ASIIMWE MUTUNGI,1,670000.00,670000.00
ADFGER,1,150001.41,150001.41
URA AGENCY NOTICE 2260019422449,1,145307.52,145307.52
DSAF,1,140237.94,140237.94
NALUNKUMA JEMIMAH RUTH,1,113000.00,113000.00
PAYMENT FOR,1,99900.15,99900.15
ADFSG,1,9500.16,9500.16
AC XFR FROM GL 0603 TO 3500,1,88.50,88.50
9030027282317:INT.COLL:01-03-2026 TO 31-03-2026,1,1.61,1.61


### Patterns

In [34]:
q2 = """
SELECT
    gm.ACID,
    gm.FORACID,
    gm.ACCT_NAME,
    gm.CIF_ID,
    gm.SOL_ID,
    gm.ACCT_OPN_DATE,
    gm.SCHM_TYPE,
    gm.SCHM_CODE,
    gm.ACCT_CRNCY_CODE,

    hd.TRAN_DATE,
    hd.TRAN_ID,
    hd.TRAN_AMT,
    hd.TRAN_PARTICULAR,
    hd.TRAN_RMKS,
    hd.PART_TRAN_TYPE,
    hd.TRAN_TYPE,
    hd.TRAN_SUB_TYPE,
    hd.RCRE_TIME,
    hd.RPT_CODE

FROM RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd

JOIN RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm
ON hd.ACID = gm.ACID

WHERE
    gm.ACCT_CRNCY_CODE = 'USD'
    AND gm.ACCT_OPN_DATE >= '2024-01-01'
    AND gm.ACCT_CLS_FLG = 'N'
"""

In [ ]:
df = pd.read_sql(query, conn)